In [30]:
from Utils import load_json_files, tfidf_similarity, bert_similarity, preprocess_json
import numpy as np
import os

In [31]:
os.getcwd()

'c:\\Users\\dbsxo\\WorkSpace\\Y\\logic\\Recommendation'

In [39]:

folder_path = "../syllabus/json"
json_files = load_json_files(folder_path)  # list[dict]
data = preprocess_json(json_files)  # np.ndarray[str], 원본과 같은 순서 가정

# 원본 인덱스 매핑 배열
orig_idx = np.arange(len(data))

# 1) 사용자 수강 리스트 통합
user_idx = np.array([1])
print([d["교과목명"] for d in json_files[user_idx]])
user_text = " ".join(data[user_idx])
print(user_text)
len(user_text)

['배터리속 소재 이야기']
배터리속 소재 이야기 배터리속 소재 이야기  materials inside battery materials inside battery  이차전지 4대 핵심소재의 기초 지식과 일반적 특성을 정확히 이해하고 활용하는 능력 지식활용능력 수업에서 다룬 소재들의 성능과 소재들 특성에 따른 이차전지 구동 원리에 대한 인과관계와 상관관계를 찾는 능력 창의사고역량 공동작업에 필요한 능력을 갖춰 원활한 이차전지 구동을 위한 소재별 최적 구성 방법을 위해 협조적으로 작업을 수행하는 능력 공감협동 이차전지 4대 핵심소재의 기초 지식과 일반적 특성을 정확히 이해하고 활용하는 능력 지식활용능력 수업에서 다룬 소재들의 성능과 소재들 특성에 따른 이차전지 구동 원리에 대한 인과관계와 상관관계를 찾는 능력 창의사고역량 공동작업에 필요한 능력을 갖춰 원활한 이차전지 구동을 위한 소재별 최적 구성 방법을 위해 협조적으로 작업을 수행하는 능력 공감협동 이차전지 4대 핵심소재의 기초 지식과 일반적 특성을 정확히 이해하고 활용하는 능력 지식활용능력 수업에서 다룬 소재들의 성능과 소재들 특성에 따른 이차전지 구동 원리에 대한 인과관계와 상관관계를 찾는 능력 창의사고역량 공동작업에 필요한 능력을 갖춰 원활한 이차전지 구동을 위한 소재별 최적 구성 방법을 위해 협조적으로 작업을 수행하는 능력 공감협동  미래 에너지 공급원의 역할을 담당할 이차전지를 소재에 관련하여 학습한다 특히 이차전지 소재 양극재 음극재 분리막 전해질 바인더 어떻게 발전되어 왔는지를 학습한다 미래 에너지 공급원의 역할을 담당할 이차전지를 소재에 관련하여 학습한다 특히 이차전지 소재 양극재 음극재 분리막 전해질 바인더 어떻게 발전되어 왔는지를 학습한다 미래 에너지 공급원의 역할을 담당할 이차전지를 소재에 관련하여 학습한다 특히 이차전지 소재 양극재 음극재 분리막 전해질 바인더 어떻게 발전되어 왔는지를 학습한다  이차전지 소재의 발전은 어떻게 이루어졌을까 강의소개 신재생에너지와 에너지 저장장치 전지의 역사 이

2893

In [38]:
from sentence_transformers import SentenceTransformer

# 1️⃣ 모델 로드
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)
tokenizer = model.tokenizer

# 2️⃣ 입력 문장 (당신의 user_text)
folder_path = "../syllabus/json"
json_files = load_json_files(folder_path)
data = preprocess_json(json_files)
user_idx = np.array([1])
user_text = " ".join(data[user_idx])

# 3️⃣ 토큰화 및 길이 확인
tokens = tokenizer.encode(user_text, add_special_tokens=True)
max_len = model.get_max_seq_length()

print(f"토큰 개수: {len(tokens)} / 최대 허용 길이: {max_len}")
if len(tokens) > max_len:
    print(f"⚠️ 초과된 토큰 수: {len(tokens) - max_len}")
    print("잘려나간 마지막 토큰 근처 예시:")
    print(tokens[max_len-10:max_len+10])  # 잘리는 구간 주변 토큰 보기


Token indices sequence length is longer than the specified maximum sequence length for this model (1618 > 128). Running this sequence through the model will result in indexing errors


토큰 개수: 1618 / 최대 허용 길이: 128
⚠️ 초과된 토큰 수: 1490
잘려나간 마지막 토큰 근처 예시:
[51692, 4156, 1504, 6547, 3776, 1190, 201, 3032, 119199, 3999, 7046, 367, 125101, 109436, 1291, 48279, 3761, 133954, 413, 136662]


In [37]:
# 실제 입력 텍스트를 자르기 전후로 비교
truncated = tokenizer.decode(tokens[:max_len])
print(f"\n입력 원문 길이(문자 기준): {len(user_text)}")
print(f"잘린 후 문장 길이(문자 기준): {len(truncated)}")


입력 원문 길이(문자 기준): 5661
잘린 후 문장 길이(문자 기준): 283


In [28]:

# 2) data/매핑 동시 삭제 후, 유저 프로필 추가
data = np.delete(data, user_idx, axis=0)
orig_idx = np.delete(orig_idx, user_idx, axis=0)
data = np.concatenate([data, np.array([user_text])])
orig_idx = np.concatenate([orig_idx, np.array([-1])])  # -1 = 유저 프로필 표시
len(data)


1869

In [29]:

# 3) TF-IDF 상위 후보
sim_tfidf = tfidf_similarity(data)
user_pos = len(data) - 1
top_indices = np.argsort(sim_tfidf[user_pos])[-100:]
top_indices = np.unique(np.append(top_indices, user_pos))
for cur_pos in top_indices:
    src_idx = orig_idx[cur_pos]  # 원본 json_files 인덱스(-1이면 유저 프로필)
    name = "USER_PROFILE" if src_idx == -1 else json_files[src_idx]["교과목명"]
    print(f"{name:<30} | sim = {sim_tfidf[user_pos][cur_pos]:.3f}")

TF-IDF similarity computed in 0.09 seconds
인터내셔널 소비자와 시장                  | sim = 0.041
소비자와 유통                        | sim = 0.045
H.O.S.T. 미디어랩 2                | sim = 0.045
지속가능디자인                        | sim = 0.043
이차전지 전해질 및 분리막                 | sim = 0.062
이차전지 제조공정 이해                   | sim = 0.047
방송학개론                          | sim = 0.050
공급사슬 전략                        | sim = 0.038
오퍼레이션스 매니지먼트                   | sim = 0.040
복지정책론                          | sim = 0.041
수학종합설계                         | sim = 0.063
펌웨어개발                          | sim = 0.039
글로벌 환경거버넌스                     | sim = 0.057
기후빅데이터 프로젝트: 캡스톤 디자인           | sim = 0.045
기후 테크와 글로벌 스타트업                | sim = 0.052
직물과편성물                         | sim = 0.039
작가발언과 자료연구 2                   | sim = 0.040
경영전략                           | sim = 0.042
교육과정                           | sim = 0.039
특수교육학개론                        | sim = 0.042
교육방법 및 교육공학                    | sim = 0.050
미디어와 사회     

In [15]:

# 4) BERT 유사도 (후보만)
sim_bert = bert_similarity(data[top_indices])

BERT similarity computed in 0.02 seconds


In [16]:

# 기준 문서 위치(후보 집합 내)
center_idx = np.where(top_indices == user_pos)[0][0]


In [25]:

# 5) 상위 N개 (자기 자신 제외)
top_n = 10
order = np.argsort(sim_bert[center_idx])[::-1]  # 내림차순
print(order)
order = order[sim_bert[center_idx][order] < 0.999999]  # self 제거
top_local = order[:top_n]
order


[99 98 97  7 87 29 30 54 70 24  6 58 59 86 57 85 45 43 33 36 82 83 75 91
 76 94 40 27 71 12 74 17 64 52 35 39 80 28  4 38 21 49 11 19 48 14 18 61
 72  5 96 31 62 41 84 73 44 16 42  1 50 47 78 26 15 65 34 51 13 79 56 90
  9 77 66 25 81 69 60 88 53 37 93 89 67 55 10 23 68 32 20 46  0  8  2  3
 92 95 63 22]


array([ 7, 87, 29, 30, 54, 70, 24,  6, 58, 59, 86, 57, 85, 45, 43, 33, 36,
       82, 83, 75, 91, 76, 94, 40, 27, 71, 12, 74, 17, 64, 52, 35, 39, 80,
       28,  4, 38, 21, 49, 11, 19, 48, 14, 18, 61, 72,  5, 96, 31, 62, 41,
       84, 73, 44, 16, 42,  1, 50, 47, 78, 26, 15, 65, 34, 51, 13, 79, 56,
       90,  9, 77, 66, 25, 81, 69, 60, 88, 53, 37, 93, 89, 67, 55, 10, 23,
       68, 32, 20, 46,  0,  8,  2,  3, 92, 95, 63, 22])

In [18]:

print("==== Top 10 Similar Courses ====")
for loc in top_local:
    cur_pos = top_indices[loc]  # data 기준 인덱스
    src_idx = orig_idx[cur_pos]  # 원본 json_files 인덱스(-1이면 유저 프로필)
    name = "USER_PROFILE" if src_idx == -1 else json_files[src_idx]["교과목명"]
    print(f"{name:<30} | sim = {sim_bert[center_idx][loc]:.3f}")

==== Top 10 Similar Courses ====
이차전지 제조공정 이해                   | sim = 0.717
이차전지 설계 실습                     | sim = 0.713
이차전지 산학 프로젝트 2                 | sim = 0.713
이차전지 소재분석공학                    | sim = 0.705
이차전지개론                         | sim = 0.702
이차전지 기초화학                      | sim = 0.686
자동차 배터리                        | sim = 0.686
이차전지 전해질 및 분리막                 | sim = 0.679
이차전지 소재평가 PBL                  | sim = 0.661
이차전지 분석 PBL                    | sim = 0.651
